# Cross-Database CVE Verification Agent

**The actual production failure mode for security agents is not hallucination. It is acting on a single source.**

When a finance agent reads a fabricated headline and executes a trade, the model didn't hallucinate; it read the source faithfully and the source was wrong. Same shape applies to security: a security agent that triages a CVE based on one database can be wrong without ever fabricating anything. The fix is corroboration across independent sources.

This notebook builds a small agent that uses [TensorFeed.ai](https://tensorfeed.ai)'s hosted MCP server to compose three independent security databases (MITRE CVE List, CISA Known Exploited Vulnerabilities, FIRST.org EPSS) for any CVE, in one tool-use loop.

**Why TensorFeed?** TF is a free MCP server (`https://tensorfeed.ai/api/mcp`) that exposes 17 tools across AI news, model pricing, security advisories, SEC filings, FDA regulatory data, and energy/macro indicators. No auth required for the tools we use here. License: most underlying data is US Government public domain; commercial redistribution permitted; attribution preserved on every response.

**What you'll see by the end:**
- An agent that takes a CVE id and returns a severity + exploitation + ecosystem fact card with explicit corroboration
- An agent that triages a list of CVEs by composite risk


## Prerequisites

- Python 3.11+
- An Anthropic API key in the `ANTHROPIC_API_KEY` environment variable
- The `anthropic` and `requests` packages

```bash
pip install anthropic requests
```

No TensorFeed account or API key needed — the three tools used in this notebook are part of TensorFeed's free tier.


In [ ]:
import os
import json
import requests
from anthropic import Anthropic

api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("Set ANTHROPIC_API_KEY in your environment first.")

client = Anthropic(api_key=api_key)
MODEL = "claude-haiku-4-5"
TENSORFEED_MCP_URL = "https://tensorfeed.ai/api/mcp"


## Calling the TensorFeed MCP server

The TensorFeed MCP server speaks JSON-RPC 2.0 over HTTP. We define a thin wrapper that calls a tool by name and returns the parsed result. In a real Claude Cowork or Claude Code session, the MCP plumbing happens automatically; we expose the wire here for clarity.


In [ ]:
def call_tensorfeed_tool(tool_name: str, arguments: dict) -> dict:
    """Call a tool on the TensorFeed MCP server via JSON-RPC 2.0."""
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {"name": tool_name, "arguments": arguments},
    }
    resp = requests.post(TENSORFEED_MCP_URL, json=payload, timeout=20)
    resp.raise_for_status()
    body = resp.json()
    if "error" in body:
        return {"ok": False, "error": body["error"]}
    # tools/call returns a content array of text blocks; we parse the first JSON block.
    content = body.get("result", {}).get("content", [])
    if not content:
        return {"ok": False, "error": "empty_content"}
    text = content[0].get("text", "{}")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"ok": False, "error": "json_decode", "raw": text[:200]}


# Quick smoke test
sample = call_tensorfeed_tool("get_cve_record", {"cve_id": "CVE-2024-3094"})
print(f"ok={sample.get('ok')}, source={sample.get('source')}")


## Tool definitions for Claude

We expose three TF tools to Claude as Anthropic API tool schemas. The descriptions tell Claude when to use each one; the input schemas constrain what Claude can pass.


In [ ]:
TOOLS = [
    {
        "name": "get_cve_record",
        "description": (
            "Look up a single CVE Record v5.2 from the MITRE CVE List by ID. "
            "Returns severity (CVSS), CWE classifications, affected vendors/products, references. "
            "Use first to confirm a CVE exists and get the canonical record."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "cve_id": {
                    "type": "string",
                    "description": "CVE identifier (e.g. CVE-2024-3094)",
                },
            },
            "required": ["cve_id"],
        },
    },
    {
        "name": "get_kev_catalog",
        "description": (
            "Get the CISA Known Exploited Vulnerabilities catalog. "
            "Use to check whether the catalog contains a specific CVE (presence indicates "
            "active exploitation in the wild). Set limit higher when looking for a specific CVE."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "limit": {
                    "type": "number",
                    "description": "How many recent entries to return (1-50)",
                    "default": 50,
                },
            },
        },
    },
    {
        "name": "get_epss_score",
        "description": (
            "Get the FIRST.org EPSS exploitation-likelihood probability for a single CVE. "
            "Returns a daily-updated probability that the CVE will be exploited in the next 30 days "
            "plus a percentile rank. EPSS is decision-useful for triage in a way CVSS is not."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "cve_id": {
                    "type": "string",
                    "description": "CVE identifier",
                },
            },
            "required": ["cve_id"],
        },
    },
]


## The agent loop

The loop is the standard Anthropic API tool-use pattern:

1. Send the user's prompt with the tool catalog.
2. If Claude responds with `tool_use`, run the tool and feed the result back as a `tool_result`.
3. Repeat until Claude responds without a tool call (final answer).

Claude decides which TF tools to call. The model knows from the tool descriptions that to verify a CVE it should typically hit MITRE first (does it exist?), then KEV (is it being exploited?), then EPSS (how likely is exploitation?).


In [ ]:
def run_agent(user_prompt: str, max_turns: int = 8) -> dict:
    """Run the verification agent. Returns the final assistant message + tool trace."""
    messages = [{"role": "user", "content": user_prompt}]
    trace = []

    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            tools=TOOLS,
            messages=messages,
            system=(
                "You are a security analyst that verifies CVEs across multiple "
                "independent databases before answering. For any CVE the user asks "
                "about, call get_cve_record + get_kev_catalog + get_epss_score, then "
                "summarize: severity_band, exploited_in_wild (true if KEV has the "
                "CVE), epss_probability, and a one-sentence triage recommendation. "
                "Always include a 'confirmed_by' list of which databases had data."
            ),
        )

        # Append the assistant turn so the next iteration sees the tool_use blocks.
        messages.append({"role": "assistant", "content": resp.content})

        if resp.stop_reason != "tool_use":
            return {"final": resp, "trace": trace, "turns": turn + 1}

        # Run each tool_use block and feed the results back as tool_result blocks.
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                trace.append({"tool": block.name, "input": block.input})
                result = call_tensorfeed_tool(block.name, block.input)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)[:8000],
                    }
                )
        messages.append({"role": "user", "content": tool_results})

    return {"final": None, "trace": trace, "turns": max_turns, "stopped": "max_turns_hit"}


## Demo 1: Verify a single CVE

CVE-2024-3094 (the XZ backdoor, March 2024) is a useful test case: it has a complete MITRE record, EPSS data, and known-exploited status varies by snapshot. The agent should consult all three sources and report whichever subset has data.


In [ ]:
result = run_agent(
    "Verify CVE-2024-3094 across multiple databases and tell me how to triage it."
)

print(f"Turns used: {result['turns']}")
print(f"Tools called: {[t['tool'] for t in result['trace']]}")
print()
if result.get("final"):
    for block in result["final"].content:
        if block.type == "text":
            print(block.text)


## Demo 2: Triage three CVEs by composite risk

A more realistic agent task: given a list of CVEs that just landed in your security feed, decide which to patch first. The agent fetches data for each and ranks them.


In [ ]:
triage_result = run_agent(
    """I have three CVEs to triage. Look up each across MITRE / KEV / EPSS and rank
them by which to patch first. Brief reasoning for each.

CVEs:
- CVE-2024-3094
- CVE-2023-44487
- CVE-2024-21626""",
    max_turns=15,
)

print(f"Turns used: {triage_result['turns']}")
print(f"Total tool calls: {len(triage_result['trace'])}")
print()
if triage_result.get("final"):
    for block in triage_result["final"].content:
        if block.type == "text":
            print(block.text)


## What just happened

Claude composed three independent TensorFeed tools into a verification loop with no extra orchestration code on our side. The agent:

1. Recognized that "verify" implies cross-source corroboration.
2. Sequenced the calls (MITRE first to confirm existence, KEV for exploitation, EPSS for likelihood).
3. Surfaced explicit `confirmed_by` so the user can audit which databases backed the answer.

For the triage task, Claude fanned out across three CVEs (~9 tool calls total) and produced a ranked list. No hallucinated CVE ids, no made-up severity scores; everything is sourced.

## Going further: TensorFeed's premium one-call composition

TensorFeed also exposes a single premium endpoint that does this composition server-side: `/api/premium/security/verified/{cve_id}`. It joins MITRE + KEV + EPSS + OSV.dev + CISA Vulnrichment into one fact card with `confirmed_by` and `corroboration_count` fields, returning ~6,000 saved tokens per call vs the multi-tool approach. The premium endpoint requires a bearer token purchased via x402 V2 on Base mainnet ($0.02/credit) at https://tensorfeed.ai/developers/agent-payments.

For most agent workloads the free three-tool composition shown above is fine; reach for the premium endpoint when you're verifying CVEs at scale (large vuln scan output, continuous monitoring, RAG indexing).

## Resources

- TensorFeed.ai: https://tensorfeed.ai
- Endpoint catalog: https://tensorfeed.ai/api/meta
- Agent-friendly entry doc: https://tensorfeed.ai/llms.txt
- Source code (public): https://github.com/RipperMercs/tensorfeed
